# **MÓDULO 26 - Projeto Final do Aprofundamento de Analytics**

Bem-vindos ao Projeto de Dashboard de E-commerce! Este projeto é uma oportunidade para vocês aplicarem habilidades essenciais de análise de dados em um cenário prático e realista. Vocês irão trabalhar com um conjunto de dados de transações de clientes de uma loja virtual, distribuídos em duas tabelas distintas. O objetivo final é construir um dashboard interativo que facilite a visualização e análise das informações relevantes do e-commerce, utilizando ferramentas como Looker Studio ou Power BI.

**Objetivo do Projeto:**

Tratamento de Dados: Realizar a junção (JOIN) de duas tabelas utilizando SQL para consolidar as informações.
Análise de Dados: Exportar os dados resultantes para um arquivo CSV.
Visualização de Dados: Desenvolver um dashboard interativo e informativo para visualização das principais métricas e insights do e-commerce.

**Tabelas Disponibilizadas:**

**Tabela de Transações:** Contém os registros de transações realizadas pelos clientes, incluindo detalhes como ID da transação, valor e outros.


**Tabela de Dados Pessoais:** Contém as informações pessoais dos clientes, como ID do cliente, nome, genero, cidade, etc.

**Chave de Ligação:** As tabelas se relacionam através da coluna ID_CLIENT, que é a chave identificadora dos clientes.

# Etapas do Projeto:

1. Realizar um JOIN SQL nas duas tabelas, unificando as informações através da coluna ID_CLIENT. Você deve justificar a escolha do JOIN (Inner/ Left/ Right ou Full).

2. Exportar os dados consolidados resultantes do JOIN para um arquivo CSV.

3. Utilizar Looker Studio ou Power BI para importar o arquivo CSV.

4. Criar visualizações interativas que apresentem métricas importantes, como total de vendas, número de transações, distribuição geográfica dos clientes, perfil demográfico dos clientes, entre outros.

Abaixo temos a configuração do ambiente SQL:

In [1]:
import sqlite3
import pandas as pd
import re

In [2]:
# Carregar os arquivos originais
df_transacoes = pd.read_csv("TB_TRANSACOES_PROJETO_ECOMM.csv", delimiter=';')
df_clientes = pd.read_csv("TB_CLIENTES_PROJETO_ECOMM.csv", delimiter=';')

In [3]:
# Criar banco SQLite
conn = sqlite3.connect('projeto.db')
df_transacoes.to_sql('tb_transacoes', conn, index=False, if_exists='replace')
df_clientes.to_sql('tb_clientes', conn, index=False, if_exists='replace')

175

In [4]:
# Função para executar consultas SQL e retornar o resultado como um DataFrame
def run_query(query):
    return pd.read_sql_query(query, conn)

In [5]:
query = """
SELECT 
    c.state_name,
    c.First_name,
    c.Gender,
    c.Job_Title,
    c.Id_client,
    t.Category,
    t.Price,
    t.`Card Type`
FROM tb_clientes c
INNER JOIN tb_transacoes t 
    ON c.Id_client = t.id_client
"""

result_df = run_query(query)

print(f"✅ Linhas após INNER JOIN: {len(result_df):,}")
print(result_df.columns.tolist())

✅ Linhas após INNER JOIN: 296
['state_name', 'First_name', 'Gender', 'Job_Title', 'Id_client', 'Category', 'Price', 'Card Type']


Justifique a escolha do JOIN:

Escolhi INNER JOIN porque o objetivo do projeto é analisar o comportamento de compra dos clientes (perfil demográfico + transações). Só faz sentido trazer registros onde existe tanto o cliente quanto a transação.
Observação importante: notei que a tabela de transações tem alguns id_client acima de 175 (ex: 360 a 367) que não existem na tabela de clientes. Com INNER JOIN esses registros foram naturalmente excluídos (evitando dados incompletos no dashboard). Se quiséssemos forçar todas as transações, poderíamos usar LEFT JOIN partindo da tabela de transações, mas perderíamos o perfil completo do cliente. INNER foi a escolha mais adequada e “limpa” para o escopo do projeto.

Exportando o arquivo como CSV:

In [45]:
# ====================== TRATATIVA COMPLETA ======================

# 1. Padronizar nomes para snake_case
def to_snake_case(name):
    name = name.lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

result_df.columns = [to_snake_case(col) for col in result_df.columns]

# 2. CORREÇÃO DO ERRO que você teve
# Forçamos a coluna price para string ANTES de fazer replace
result_df['price'] = result_df['price'].astype(str).str.replace(',', '.').astype(float)

result_df = result_df.rename(columns={
    'state_name': 'estado',
    'first_name': 'nome',
    'gender': 'genero',
    'job_title': 'cargo',
    'id_client': 'id_cliente',
    'category': 'categoria',
    'price': 'preco',
    'card_type': 'tipo_cartao'
})

# 4. Verificação final
print("✅ Colunas finais (em português):")
print(result_df.columns.tolist())
print("\nPrévia dos dados limpos:")
print(result_df.head(3))
print("\nTipos de dados:")
print(result_df.dtypes)

✅ Colunas finais (em português):
['estado', 'nome', 'genero', 'cargo', 'id_cliente', 'categoria', 'preco', 'tipo_cartao']

Prévia dos dados limpos:
  estado     nome genero                         cargo  id_cliente categoria  \
0     TX  Domingo   Male  Structural Analysis Engineer           1  Outdoors   
1     MI  Russell   Male            Speech Pathologist           2   Grocery   
2     AL   Kimble   Male           Account Coordinator           3     Music   

    preco tipo_cartao  
0   16.97  mastercard  
1  143.39  mastercard  
2   37.64  mastercard  

Tipos de dados:
estado          object
nome            object
genero          object
cargo           object
id_cliente       int64
categoria       object
preco          float64
tipo_cartao     object
dtype: object


In [46]:
# Exemplo de LTV (Lifetime Value) por cliente – usando GROUP BY + AS
query_ltv = """
SELECT 
    t.id_client AS id_cliente,
    SUM(CAST(REPLACE(t.Price, ',', '.') AS FLOAT)) AS total_gasto_ltv,
    COUNT(*) AS qtd_transacoes,
    ROUND(AVG(CAST(REPLACE(t.Price, ',', '.') AS FLOAT)), 2) AS ticket_medio
FROM tb_transacoes t
INNER JOIN tb_clientes c ON c.Id_client = t.id_client
GROUP BY t.id_client
ORDER BY total_gasto_ltv DESC
LIMIT 10
"""

ltv_df = run_query(query_ltv)
print(ltv_df)

   id_cliente  total_gasto_ltv  qtd_transacoes  ticket_medio
0          38           322.52               3        107.51
1         117           313.34               3        104.45
2          39           310.70               3        103.57
3         158           280.90               2        140.45
4          50           274.53               2        137.26
5          46           269.26               2        134.63
6         119           263.60               2        131.80
7         141           263.05               2        131.53
8          29           251.35               2        125.67
9         121           250.81               2        125.41


In [47]:
query_ltv = """
SELECT 
    t.id_client AS id_cliente,
    c.First_name AS nome,
    SUM(CAST(REPLACE(t.Price, ',', '.') AS FLOAT)) AS total_gasto_ltv,
    COUNT(*) AS qtd_transacoes,
    ROUND(AVG(CAST(REPLACE(t.Price, ',', '.') AS FLOAT)), 2) AS ticket_medio
FROM tb_transacoes t
INNER JOIN tb_clientes c ON c.Id_client = t.id_client
GROUP BY t.id_client, c.First_name
ORDER BY total_gasto_ltv DESC
LIMIT 10
"""

ltv_df = run_query(query_ltv)
print(ltv_df)

   id_cliente      nome  total_gasto_ltv  qtd_transacoes  ticket_medio
0          38       Rab           322.52               3        107.51
1         117    Portie           313.34               3        104.45
2          39     Codie           310.70               3        103.57
3         158     Jarib           280.90               2        140.45
4          50   Sergent           274.53               2        137.26
5          46  Demetris           269.26               2        134.63
6         119   Joaquin           263.60               2        131.80
7         141     Britt           263.05               2        131.53
8          29     Jonas           251.35               2        125.67
9         121    Bartel           250.81               2        125.41


Decidi utilizar query_ltv, devido a algumas sugestões dos tutores anteriores a fim de facilitar a interpretação por outras pessoas que lerem o meu relatório.
    

**Dicas para o projeto:**
- Se atente que, como o mesmo cliente realiza mais de 1 transação quando você for trazer alguma métrica relacionada a dados do cliente terá que utilizar o distinct para criar essas métricas no dashboard, se não acabará tendo os dados repetidos.

- Análise sua tabela, entenda a dimensão dos dados, no excel, antes de enviar para o Powerbi ou Looker Studio.

- Tente montar preveamente um roteiro de quais métricas e visualizações irá colocar no dashboard, isso tornará seu processo mais rápido.

- Qualquer dificuldade para subir sua base para as ferramentas de visualização envie a base e o erro encontrado para que os tutores possam te ajudar.

In [48]:
result_df.to_csv('dados_ecommerce_final.csv', index=False)
print("✅ Arquivo 'dados_ecommerce_final.csv' exportado com sucesso!")

✅ Arquivo 'dados_ecommerce_final.csv' exportado com sucesso!
